# IRIS Classification With Learned Superposition Activations

This notebook keeps the notebook 06 story intact, but now the dataset preparation and experiment/reporting flow come from shared helpers so later benchmark notebooks stay aligned with it.

The hidden activations are still learned from trainable quantum superpositions, optimization still uses the exact reconstructed activation profiles, and selected learned activations are still validated through measurements after training.

In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd().parent):
    if (_p / "qfun").is_dir():
        _root = str(_p.resolve())
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break

from qfun.datasets import load_classification_dataset, prepare_classification_split
from qfun.qfan._classification_benchmarks import (
    build_comparison_rows,
    display_baseline_suite,
    display_quantum_result,
    print_comparison_table,
    print_split_summary,
    plot_training_diagnostics,
    run_default_baseline_suite,
    run_quantum_experiment,
)

## Config

These defaults match the current notebook 06 setup, and the training-process snapshots remain recorded every 5 epochs.

In [ ]:
data_seed = 7
test_size = 0.25

hidden_units = 4
n_qubits = 5
steps = 30
learning_rate = 0.05
log_every = 5
snapshot_interval = 5
eval_shots = 5_000

## 1. Load And Standardize IRIS

We use the full 3-class IRIS dataset, standardize the four input features, and keep a stratified train/test split so each class is represented in both sets.

In [ ]:
iris_dataset = load_classification_dataset("iris")
iris_split = prepare_classification_split(
    iris_dataset,
    test_size=test_size,
    seed=data_seed,
    standardize=True,
)
class_names = iris_split.target_names
print_split_summary(iris_dataset.name, iris_split)

## 2. Baselines

Before using learned superposition activations, we fit the same two classical baselines as before: multinomial logistic regression and a small sklearn MLP.

In [ ]:
baseline_results = run_default_baseline_suite(iris_split, seed=data_seed)
display_baseline_suite(baseline_results, class_names)

## 3. Standard Superposition Activations

In the standard mode, each hidden unit learns a nonnegative activation profile from a single real amplitude vector of length `2**n_qubits`.

In [ ]:
standard_result = run_quantum_experiment(
    "standard",
    label="Standard superposition activations",
    split=iris_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
)
display_quantum_result(standard_result, class_names)

### Standard Training Process (Snapshots)

The loss curve follows notebook 04: every 5 epochs we mark a snapshot, print a compact step/loss table, and show how the first few learned activations evolve.

In [ ]:
plot_training_diagnostics(standard_result)

## 4. Mode A Signed Superposition Activations

Mode A learns signed activations by training one extended state on `n_qubits + 1` wires and reconstructing the profile from `p_pos - p_neg`.

In [ ]:
mode_a_result = run_quantum_experiment(
    "mode_a",
    label="Mode A superposition activations",
    split=iris_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
)
display_quantum_result(mode_a_result, class_names)

### Mode A Training Process (Snapshots)

These curves are the exact signed activations reconstructed from the learned ancilla-based states every 5 epochs.

In [ ]:
plot_training_diagnostics(mode_a_result)

## 5. Mode B Signed Superposition Activations

Mode B learns two nonnegative channels plus mixing weights `z_plus` and `z_minus`, then forms the signed activation `q = z_plus * p_plus - z_minus * p_minus`.

In [ ]:
mode_b_result = run_quantum_experiment(
    "mode_b",
    label="Mode B superposition activations",
    split=iris_split,
    hidden_units=hidden_units,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
)
display_quantum_result(mode_b_result, class_names)

### Mode B Training Process (Snapshots)

The activation curves below show the signed reconstructions built from the two learned channels every 5 epochs.

In [ ]:
plot_training_diagnostics(mode_b_result)

## 6. Final Comparison

This puts the classical baselines and all three superposition-activation classifiers on the same IRIS test split.

In [ ]:
comparison_rows = build_comparison_rows(
    baseline_results,
    [standard_result, mode_a_result, mode_b_result],
)
print_comparison_table(comparison_rows)